# 08i — Explain (Attention + GNNExplainer, DGCNN / SortPooling Readout)

Post-hoc explainability for `07i`'s trained checkpoints. **Requires
`07i` to have already been run** -- this notebook loads its saved
`{tag}_best_model.pt` / `{tag}_best_model_stats.pt` per scenario, it
does not train anything itself. Structurally identical to `08g`
(same two techniques, same point-selection logic, same normalization
handling) -- see `08g`'s intro for the shared mechanics. This intro
covers only what's specific to explaining a `readout="dgcnn"` model.

**The question this notebook exists to answer.** `docs/07g_07i_architecture.md`
§4 flags that `DGCNNReadout` gives **no anchor guarantee** -- unlike
`07g`'s `pool_anchor`, which always concatenates the incident/ego node's
own embedding into the graph vector, DGCNN's SortPooling ranks every
node by salience and keeps only the top-`k`; the anchor can rank outside
that cutoff and be dropped entirely. This was flagged as a real
possibility, not assumed either way. GNNExplainer's learned
node-importance mask gives a direct way to check: **does the anchor
node's importance collapse for `07i` relative to `07g`, and does that
correlate with wrong predictions (FP/FN)?**

**`k` reconstruction.** `DGCNNReadout`'s Conv1d/Linear layer shapes
depend on `k`, which `07i` computed dynamically from real data (the
40th-percentile rule, per encoder) rather than reading it from a fixed
config value -- so this notebook must recompute the SAME `k` values
before it can even construct the right architecture to load
`best_model.pt`'s state_dict into. The diagnostic cell below is copied
verbatim from `07i` (same sample, same `random_state=42`) -- as long as
the underlying dataset hasn't changed, this reproduces the exact `k`
`07i` actually trained with.

**Attention extraction is UNCHANGED by the readout swap** -- both
branches use `conv_type="gatv2"`, and attention lives in the
message-passing layers, not the readout. Any difference you see between
`08g` and `08i`'s attention-weight patterns reflects the readout
indirectly shaping what the encoder learns to attend to, not a direct
mechanical effect.

GPU recommended for the GNNExplainer optimization loop.

**v2**: adds a feature-level explanation section at the end -- see that section's own markdown intro for what changed and why.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched src/ files until pushed to GitHub.
# Skip this cell once the repo itself is updated -- needs explain.py,
# train.py (best_model_stats.pt persistence), models.py
# (DGCNNReadout + readout="dgcnn"), plus graph_datasets.py,
# unified_graph.py, baseline_features.py, evaluate.py, plot_history.py.
from google.colab import files
import shutil

print("Upload explain.py, train.py, models.py, graph_datasets.py, unified_graph.py, "
      "baseline_features.py, evaluate.py, plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm seaborn matplotlib

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_dgcnn_comparison.yaml") as f:
    model_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
COMBINED_PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
# MUST match 07i's own dir names exactly.
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_dgcnn_comparison"
METRICS_DIR = OUTPUTS_DIR / "metrics_dgcnn_comparison"
assert CHECKPOINT_DIR.exists(), f"{CHECKPOINT_DIR} not found -- run 07i first."
EXPLAIN_DIR = OUTPUTS_DIR / "explain_dgcnn_comparison"
EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)
# 08g's dir, read-only here -- only used later by the comparison cell.
CAPACITY_REVISION_EXPLAIN_DIR = OUTPUTS_DIR / "explain_capacity_revision"

device = "cuda" if torch.cuda.is_available() else "cpu"
HEAD_DEPTH = model_cfg.get("head_depth", "mlp2")
CONV_TYPE = model_cfg.get("conv_type", "gatv2")
READOUT = model_cfg.get("readout", "dgcnn")

# SET THIS YOURSELF -- how many points per confusion-matrix category
# (TP/TN/FP/FN) to explain, per scenario. No default is assumed here on
# purpose: each explained point costs one GNNExplainer optimization run
# (GNNEXPLAINER_EPOCHS steps), so the right number depends on how much
# time/compute you want to spend and how statistically defensible you
# need the resulting importance averages to be. Use the SAME value here
# as in 08g if you want the normal-vs-ablation and DGCNN-vs-pool_anchor
# comparisons to rest on comparably-sized samples.
N_PER_CATEGORY = None
assert N_PER_CATEGORY is not None, "Set N_PER_CATEGORY above before running the rest of this notebook."

GNNEXPLAINER_EPOCHS = 100
GNNEXPLAINER_LR = 0.05
EXPLAIN_SEED = 42

print("Device:", device, "| head_depth:", HEAD_DEPTH, "| readout:", READOUT)
print("Reading checkpoints from:", CHECKPOINT_DIR)
print("Writing explanations to:", EXPLAIN_DIR)
print(f"N_PER_CATEGORY={N_PER_CATEGORY}, gnnexplainer_epochs={GNNEXPLAINER_EPOCHS}")

In [ ]:
import json
import copy
import random
import pandas as pd
import graph_datasets as ds
import models
import explain
import unified_graph as ug

SVG_DIR = COMBINED_PROCESSED_DIR / "svg_graphs"
TVG_DIR = COMBINED_PROCESSED_DIR / "tvg_graphs"
INDEX_PATH = COMBINED_PROCESSED_DIR / "dataset_index.parquet"
index_df = pd.read_parquet(INDEX_PATH)
assert "city" in index_df.columns, (
    f"'{INDEX_PATH}' has no 'city' column -- this notebook needs 05's combined, "
    "multi-city dataset_index.parquet, not a single-city index.")
print(f"Dataset: {len(index_df)} points available for lookup")

_ref_cache_dir = INTERIM_DIR / "osm_cache" / CITIES[0]
with open(_ref_cache_dir / "highway_vocab.json") as f:
    HIGHWAY_VOCAB_SIZE = len(json.load(f))
with open(_ref_cache_dir / "building_type_vocab.json") as f:
    BUILDING_TYPE_VOCAB_SIZE = len(json.load(f))
print(f"Unified vocab (post-04b): highway={HIGHWAY_VOCAB_SIZE}, building_type={BUILDING_TYPE_VOCAB_SIZE}")

## Reconstruct `k` -- copied verbatim from 07i's own diagnostic cell

Must reproduce the exact `k` values `07i` actually trained with, or the
Conv1d/Linear layers inside `DGCNNReadout` won't match
`best_model.pt`'s saved state_dict shapes. Same sample (`n=500,
random_state=42`), same rule (`k = max(floor, p40)`) -- deterministic
given the same underlying dataset.

In [ ]:
import numpy as np
from graph_datasets import DualGraphDataset

_lookup_dataset = DualGraphDataset(index_df, SVG_DIR, TVG_DIR)

def _total_node_count(hetero_data, node_types):
    return sum(int(hetero_data[nt].x.shape[0]) for nt in node_types if nt in hetero_data.node_types)

SAMPLE_N = min(500, len(_lookup_dataset))
sample_idx = index_df.sample(n=SAMPLE_N, random_state=42).index

_tvg_node_types_no_peer = [nt for nt in models.TVG_NODE_TYPES if nt != "peer_incident"]

svg_counts, tvg_counts, unified_counts = [], [], []
for i in sample_idx:
    svg_d, tvg_d, _label, _pid = _lookup_dataset[i]
    svg_counts.append(_total_node_count(svg_d, models.SVG_NODE_TYPES))
    tvg_counts.append(_total_node_count(tvg_d, _tvg_node_types_no_peer))
    merged_d = ug.merge_svg_tvg(svg_d, tvg_d)
    unified_counts.append(_total_node_count(merged_d, models.UNIFIED_NODE_TYPES))

def _compute_k(counts, conv2_kernel, label):
    counts = np.asarray(counts)
    p40 = np.percentile(counts, 40)
    floor_k = (conv2_kernel - 1) * 2 + 2
    k = max(floor_k, int(np.ceil(p40)))
    print(f"{label:8s} | p40={p40:5.1f} | conv2_kernel={conv2_kernel} floor={floor_k:2d} -> k={k:3d}")
    return k

SVG_DGCNN_K = _compute_k(svg_counts, model_cfg.get("svg_dgcnn_conv2_kernel", 5), "SVG")
TVG_DGCNN_K = _compute_k(tvg_counts, model_cfg.get("tvg_dgcnn_conv2_kernel", 3), "TVG")
UNIFIED_DGCNN_K = _compute_k(unified_counts, model_cfg.get("unified_dgcnn_conv2_kernel", 5), "Unified")

In [ ]:
svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2,
                   cat_embed_dim=model_cfg.get("cat_embed_dim", 4), conv_type=CONV_TYPE,
                   readout=READOUT, dgcnn_k=SVG_DGCNN_K,
                   dgcnn_conv2_kernel=model_cfg.get("svg_dgcnn_conv2_kernel", 5))
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.3),
                   building_type_vocab=BUILDING_TYPE_VOCAB_SIZE, highway_vocab=HIGHWAY_VOCAB_SIZE,
                   building_type_embed_dim=model_cfg.get("building_type_embed_dim", 32),
                   highway_embed_dim=model_cfg.get("highway_embed_dim", 8), conv_type=CONV_TYPE,
                   readout=READOUT, dgcnn_k=TVG_DGCNN_K,
                   dgcnn_conv2_kernel=model_cfg.get("tvg_dgcnn_conv2_kernel", 3))
FUSION_DIM = model_cfg.get("fusion_dim", 256)
HEAD_HIDDEN = model_cfg.get("head_hidden", 256)
HEAD_DROPOUT = model_cfg.get("head_dropout", 0.3)
print("svg_kwargs:", svg_kwargs)
print("tvg_kwargs:", tvg_kwargs)
print("unified_dgcnn_k:", UNIFIED_DGCNN_K)

## Helpers: load a trained scenario's model+stats, pick which points to explain

Same logic as `08g`'s helpers, with one addition: scenario F needs
`unified_dgcnn_k` passed explicitly to `build_model()` (mirrors 07i's own
call -- see `build_model`'s `unified_dgcnn_k` override, without which
`svg_kwargs`'s `dgcnn_k` would silently win the kwarg merge).

In [ ]:
def load_scenario_model(scenario, use_ablation=False):
    tag = f"{scenario}_{HEAD_DEPTH}" + ("_ablation" if use_ablation else "")
    model_path = CHECKPOINT_DIR / f"{tag}_best_model.pt"
    stats_path = CHECKPOINT_DIR / f"{tag}_best_model_stats.pt"
    meta_path = CHECKPOINT_DIR / f"{tag}_best_model_meta.json"
    assert model_path.exists(), f"{model_path} not found -- run 07i's scenario {scenario} cell first."
    assert stats_path.exists(), (
        f"{stats_path} not found -- this checkpoint predates the best_model_stats.pt "
        f"persistence change; rerun 07i's scenario {scenario} cell with the current train.py.")

    model = models.build_model(scenario, fusion_dim=FUSION_DIM, head_depth=HEAD_DEPTH,
                                head_hidden=HEAD_HIDDEN, head_dropout=HEAD_DROPOUT,
                                use_ablation=use_ablation, svg_kwargs=svg_kwargs, tvg_kwargs=tvg_kwargs,
                                unified_dgcnn_k=UNIFIED_DGCNN_K)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=False))
    model.eval()
    stats = torch.load(stats_path, map_location="cpu", weights_only=False)
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    best_repeat = meta.get("repeat")
    return model, stats, best_repeat, tag


def pick_points_to_explain(tag, best_repeat, n_per_category=N_PER_CATEGORY, seed=EXPLAIN_SEED):
    if best_repeat is None:
        print(f"  [{tag}] no best_model_meta.json repeat recorded -- skipping point selection.")
        return []
    pred_path = CHECKPOINT_DIR / f"{tag}_history" / f"repeat{best_repeat}_test_predictions.json"
    if not pred_path.exists():
        print(f"  [{tag}] {pred_path} not found -- skipping.")
        return []
    records = json.loads(pred_path.read_text())
    rng = random.Random(seed)
    by_category = {}
    for r in records:
        by_category.setdefault(r["category"], []).append(r)
    picked = []
    for cat, rows in sorted(by_category.items()):
        sample = rng.sample(rows, min(n_per_category, len(rows)))
        picked.extend({"point_id": r["point_id"], "category": cat} for r in sample)
    return picked

## Run both explanation techniques across scenarios A-F -- normal and ablation, SEPARATELY

Two independent passes, never merged: `NORMAL_SCENARIOS` (A-F,
`use_ablation=False`) and `ABLATION_SCENARIOS` (B-F only -- `A` has no
ablation variant, matching `07i`'s own scenario cells). Each pass
produces its own record list, its own tidy DataFrame, and its own set
of output CSVs (`..._normal.csv` / `..._ablation.csv`).

In [ ]:
NORMAL_SCENARIOS = ["A", "B", "C", "D", "E", "F"]
ABLATION_SCENARIOS = ["B", "C", "D", "E", "F"]  # A has no ablation variant


def run_explanation_pass(scenarios, use_ablation):
    records = []
    label = "ablation" if use_ablation else "normal"
    for scenario in scenarios:
        print(f"\n=== [{label}] Scenario {scenario} ===")
        model, stats, best_repeat, tag = load_scenario_model(scenario, use_ablation=use_ablation)
        print(f"  loaded {tag} (best repeat={best_repeat})")

        points = pick_points_to_explain(tag, best_repeat, n_per_category=N_PER_CATEGORY)
        print(f"  explaining {len(points)} points: "
              f"{ {c: sum(1 for p in points if p['category']==c) for c in sorted(set(p['category'] for p in points))} }")

        for p in points:
            pid, cat = p["point_id"], p["category"]
            svg_raw = torch.load(SVG_DIR / f"{pid}.pt", weights_only=False)
            tvg_raw = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
            svg_norm, tvg_norm = ds.apply_normalization(copy.deepcopy(svg_raw), copy.deepcopy(tvg_raw), stats)

            try:
                recs = explain.explain_scenario_point(
                    model, scenario, svg_norm, tvg_norm, point_id=pid, category=cat,
                    gnnexplainer_epochs=GNNEXPLAINER_EPOCHS, gnnexplainer_lr=GNNEXPLAINER_LR)
                records.extend(recs)
            except Exception as e:
                print(f"    !! failed to explain {pid} ({cat}): {type(e).__name__}: {e}")
    print(f"\n[{label}] total explained (point, branch) records: {len(records)}")
    return records


normal_records = run_explanation_pass(NORMAL_SCENARIOS, use_ablation=False)
ablation_records = run_explanation_pass(ABLATION_SCENARIOS, use_ablation=True)

## Aggregate into two tidy CSVs -- normal and ablation kept separate

In [ ]:
explain_df_normal = explain.aggregate_explanations(normal_records)
explain_df_ablation = explain.aggregate_explanations(ablation_records)

explain_df_normal.to_csv(EXPLAIN_DIR / "explanations_dgcnn_comparison_normal.csv", index=False)
explain_df_ablation.to_csv(EXPLAIN_DIR / "explanations_dgcnn_comparison_ablation.csv", index=False)

print(f"Saved {len(explain_df_normal)} rows to {EXPLAIN_DIR / 'explanations_dgcnn_comparison_normal.csv'}")
print(f"Saved {len(explain_df_ablation)} rows to {EXPLAIN_DIR / 'explanations_dgcnn_comparison_ablation.csv'}")
display(explain_df_normal.head(10))
display(explain_df_ablation.head(10))

## The question this notebook exists to answer: does DGCNN actually drop the anchor? (normal vs. ablation, separately)

Unlike `08g`, `DGCNNReadout` has NO guarantee the anchor node
(`ego`/`incident`) survives SortPooling's top-`k` cutoff. This cell
computes the same anchor-importance summary `08g` does -- independently
for the normal and ablation passes -- so each can be compared directly
against `08g`'s matching pass: a lower anchor importance here
(especially on FP/FN points) would be direct empirical support for the
"no anchor guarantee" concern; a similar or higher anchor importance
would suggest the model still learns to rank the anchor highly on its
own, even without the architectural guarantee.

In [ ]:
anchor_types_normal = {"A": "ego", "B": "incident", "C_svg": "ego", "C_tvg": "incident",
                       "D_svg": "ego", "D_tvg": "incident", "E_svg": "ego", "E_tvg": "incident",
                       "F": "incident"}
anchor_types_ablation = {k: v for k, v in anchor_types_normal.items() if k != "A"}


def build_anchor_summary(explain_df, anchor_types):
    gnne_node = explain_df[(explain_df["source"] == "gnnexplainer") & (explain_df["kind"] == "node")]
    rows = []
    for scen, anchor_nt in anchor_types.items():
        sub = gnne_node[gnne_node["scenario"] == scen]
        anchor_rows = sub[sub["type"] == anchor_nt]
        other_rows = sub[sub["type"] != anchor_nt]
        if len(anchor_rows) == 0:
            continue
        row = {
            "scenario": scen, "anchor_type": anchor_nt,
            "anchor_mean_importance": anchor_rows["mean_value"].mean(),
            "other_types_mean_importance": other_rows["mean_value"].mean() if len(other_rows) else float("nan"),
            "n_points": anchor_rows["point_id"].nunique(),
        }
        # split by category too -- does the anchor's importance drop specifically on wrong predictions?
        for cat in ["TP", "TN", "FP", "FN"]:
            cat_anchor = anchor_rows[anchor_rows["category"] == cat]
            if len(cat_anchor):
                row[f"anchor_importance_{cat}"] = cat_anchor["mean_value"].mean()
        rows.append(row)
    return pd.DataFrame(rows)


def compare_anchor_against_08g(anchor_df, suffix, out_name):
    """suffix: 'normal' or 'ablation' -- picks which of 08g's matching
    anchor_importance_summary_{suffix}.csv to compare against."""
    path = CAPACITY_REVISION_EXPLAIN_DIR / f"anchor_importance_summary_{suffix}.csv"
    if not path.exists():
        print(f"08g's anchor_importance_summary_{suffix}.csv not found yet -- run 08g first to compare.")
        return None
    gatv2_df = pd.read_csv(path)
    compare = anchor_df[["scenario", "anchor_type", "anchor_mean_importance"]].merge(
        gatv2_df[["scenario", "anchor_mean_importance"]], on="scenario", suffixes=("_dgcnn", "_pool_anchor"))
    compare["anchor_importance_delta"] = (
        compare["anchor_mean_importance_dgcnn"] - compare["anchor_mean_importance_pool_anchor"])
    compare.to_csv(EXPLAIN_DIR / out_name, index=False)
    display(compare)
    n_lower = (compare["anchor_importance_delta"] < 0).sum()
    print(f"[{suffix}] DGCNN's anchor importance is LOWER than pool_anchor's on {n_lower}/{len(compare)} scenarios.")
    return compare


anchor_summary_normal_df = build_anchor_summary(explain_df_normal, anchor_types_normal)
anchor_summary_ablation_df = build_anchor_summary(explain_df_ablation, anchor_types_ablation)

anchor_summary_normal_df.to_csv(EXPLAIN_DIR / "anchor_importance_summary_normal.csv", index=False)
anchor_summary_ablation_df.to_csv(EXPLAIN_DIR / "anchor_importance_summary_ablation.csv", index=False)
display(anchor_summary_normal_df)
display(anchor_summary_ablation_df)

anchor_compare_normal_df = compare_anchor_against_08g(
    anchor_summary_normal_df, "normal", "anchor_importance_dgcnn_vs_pool_anchor_normal.csv")
anchor_compare_ablation_df = compare_anchor_against_08g(
    anchor_summary_ablation_df, "ablation", "anchor_importance_dgcnn_vs_pool_anchor_ablation.csv")

## Final report: per-scheme importance summary, and DGCNN vs. pool_anchor across ALL types

Same per-scheme summary `08g` builds (`type_importance_summary.csv` /
`type_importance_top5.csv`), plus one comparison the anchor-only check
above doesn't cover: **does SortPooling shift importance across *every*
node/edge type**, not just the anchor -- e.g. does dropping the anchor
guarantee make the model lean harder on `building`/`signage` nodes
instead?

In [ ]:
type_pivot_normal_df, type_topn_normal_df = explain.build_type_importance_report(explain_df_normal, top_n=5)
type_pivot_ablation_df, type_topn_ablation_df = explain.build_type_importance_report(explain_df_ablation, top_n=5)

type_pivot_normal_df.to_csv(EXPLAIN_DIR / "type_importance_summary_normal.csv", index=False)
type_topn_normal_df.to_csv(EXPLAIN_DIR / "type_importance_top5_normal.csv", index=False)
type_pivot_ablation_df.to_csv(EXPLAIN_DIR / "type_importance_summary_ablation.csv", index=False)
type_topn_ablation_df.to_csv(EXPLAIN_DIR / "type_importance_top5_ablation.csv", index=False)

print(f"[normal]   saved {len(type_pivot_normal_df)} summary rows, {len(type_topn_normal_df)} top-5 rows")
print(f"[ablation] saved {len(type_pivot_ablation_df)} summary rows, {len(type_topn_ablation_df)} top-5 rows")
display(type_topn_normal_df)
display(type_topn_ablation_df)


def compare_type_against_08g(type_pivot_df, suffix, out_name):
    path = CAPACITY_REVISION_EXPLAIN_DIR / f"type_importance_summary_{suffix}.csv"
    if not path.exists():
        print(f"08g's type_importance_summary_{suffix}.csv not found yet -- run 08g first to compare.")
        return None
    gatv2_df = pd.read_csv(path)
    compare = type_pivot_df.merge(
        gatv2_df[["scenario", "kind", "type", "gnnexplainer", "attention"]],
        on=["scenario", "kind", "type"], suffixes=("_dgcnn", "_pool_anchor"), how="inner")
    compare["gnnexplainer_delta"] = compare["gnnexplainer_dgcnn"] - compare["gnnexplainer_pool_anchor"]
    compare = compare.sort_values("gnnexplainer_delta")
    compare.to_csv(EXPLAIN_DIR / out_name, index=False)
    display(compare)
    print(f"[{suffix}] saved {len(compare)} rows to {EXPLAIN_DIR / out_name} "
          "(sorted by gnnexplainer_delta: most negative = DGCNN under-weights that type "
          "relative to pool_anchor the most).")
    return compare


type_compare_normal_df = compare_type_against_08g(
    type_pivot_normal_df, "normal", "type_importance_dgcnn_vs_pool_anchor_normal.csv")
type_compare_ablation_df = compare_type_against_08g(
    type_pivot_ablation_df, "ablation", "type_importance_dgcnn_vs_pool_anchor_ablation.csv")

## Figures: top-5 importance per scenario, seaborn, L-shaped (despined) axes

Same convention as `08g`: `gnnexplainer` only (attention is dominated by
node-degree ceiling effects, not a real importance signal -- see
`08g`'s equivalent cell), L-shaped axes via `sns.despine()` (only
left+bottom spines kept), muted palette.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("ticks")
sns.set_context("paper")
PALETTE = sns.color_palette("muted")
FIG_DIR = EXPLAIN_DIR


def plot_top_importance_grid(topn_df, title, filename, source="gnnexplainer"):
    """One horizontal bar subplot per scenario, top-5 (scenario, source)
    rows from build_type_importance_report's topn_df. L-shaped axes
    (sns.despine: only left+bottom spines kept)."""
    sub = topn_df[topn_df["source"] == source].copy()
    scenarios = sorted(sub["scenario"].unique())
    n = len(scenarios)
    if n == 0:
        print(f"No '{source}' rows in {title} -- skipping figure.")
        return
    ncols = 3
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), squeeze=False)

    for i, scen in enumerate(scenarios):
        ax = axes[i // ncols][i % ncols]
        scen_df = sub[sub["scenario"] == scen].sort_values("mean_importance")
        labels = [f"{r.kind}:{str(r.type)[:28]}" for r in scen_df.itertuples()]
        ax.barh(labels, scen_df["mean_importance"], color=PALETTE[0])
        ax.set_title(scen, fontsize=11, fontweight="bold")
        ax.set_xlim(0, 1)
        ax.set_xlabel("mean gnnexplainer importance")
        ax.tick_params(axis="y", labelsize=8)
        sns.despine(ax=ax)

    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches="tight")
    print(f"Saved figure to {FIG_DIR / filename}")
    plt.show()


plot_top_importance_grid(type_topn_normal_df, "Top-5 node/edge importance per scenario (normal, DGCNN)",
                          "fig_top_importance_normal.png")
plot_top_importance_grid(type_topn_ablation_df, "Top-5 node/edge importance per scenario (ablation, DGCNN)",
                          "fig_top_importance_ablation.png")

## Figures: anchor vs. other node types (DGCNN), and DGCNN-vs-pool_anchor deltas

In [ ]:
def plot_anchor_comparison(anchor_df, title, filename):
    if anchor_df.empty:
        print(f"No rows for {title} -- skipping figure.")
        return
    melted = anchor_df.melt(id_vars=["scenario"],
                             value_vars=["anchor_mean_importance", "other_types_mean_importance"],
                             var_name="group", value_name="importance")
    melted["group"] = melted["group"].map({"anchor_mean_importance": "anchor",
                                            "other_types_mean_importance": "other types (avg)"})
    fig, ax = plt.subplots(figsize=(7, 0.55 * len(anchor_df) + 1.5))
    sns.barplot(data=melted, y="scenario", x="importance", hue="group", ax=ax, palette="muted")
    ax.set_xlim(0, 1)
    ax.set_xlabel("mean gnnexplainer importance")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(title=None, frameon=False, loc="lower right")
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches="tight")
    print(f"Saved figure to {FIG_DIR / filename}")
    plt.show()


def plot_dgcnn_vs_pool_anchor_delta(compare_df, title, filename):
    """Diverging horizontal bar: which node/edge types does DGCNN weight
    MORE (positive) or LESS (negative) than pool_anchor, per scenario."""
    if compare_df is None or compare_df.empty:
        print(f"No rows for {title} -- skipping figure.")
        return
    top = pd.concat([compare_df.nsmallest(10, "gnnexplainer_delta"),
                      compare_df.nlargest(10, "gnnexplainer_delta")]).drop_duplicates()
    top = top.sort_values("gnnexplainer_delta")
    labels = [f"{r.scenario}:{r.kind}:{str(r.type)[:22]}" for r in top.itertuples()]
    colors = [PALETTE[3] if v < 0 else PALETTE[2] for v in top["gnnexplainer_delta"]]

    fig, ax = plt.subplots(figsize=(7, 0.35 * len(top) + 1.5))
    ax.barh(labels, top["gnnexplainer_delta"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("gnnexplainer_delta (DGCNN minus pool_anchor)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.tick_params(axis="y", labelsize=8)
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches="tight")
    print(f"Saved figure to {FIG_DIR / filename}")
    plt.show()


plot_anchor_comparison(anchor_summary_normal_df, "Anchor vs. other node types -- normal (DGCNN)", "fig_anchor_normal.png")
plot_anchor_comparison(anchor_summary_ablation_df, "Anchor vs. other node types -- ablation (DGCNN)", "fig_anchor_ablation.png")

plot_dgcnn_vs_pool_anchor_delta(type_compare_normal_df, "DGCNN vs. pool_anchor -- normal (top 10 largest deltas)",
                                 "fig_dgcnn_vs_pool_anchor_delta_normal.png")
plot_dgcnn_vs_pool_anchor_delta(type_compare_ablation_df, "DGCNN vs. pool_anchor -- ablation (top 10 largest deltas)",
                                 "fig_dgcnn_vs_pool_anchor_delta_ablation.png")

In [ ]:
print("08i explainability run complete.")
print(f"[normal]   {explain_df_normal['point_id'].nunique()} unique points across "
      f"{explain_df_normal['scenario'].nunique()} scenario/branch tags.")
print(f"[ablation] {explain_df_ablation['point_id'].nunique()} unique points across "
      f"{explain_df_ablation['scenario'].nunique()} scenario/branch tags.")
print(f"k values used (reconstructed to match 07i): svg={SVG_DGCNN_K} tvg={TVG_DGCNN_K} unified={UNIFIED_DGCNN_K}")
print()
print("Normal outputs:   explanations_dgcnn_comparison_normal.csv, anchor_importance_summary_normal.csv,")
print("                  type_importance_summary_normal.csv, type_importance_top5_normal.csv,")
print("                  anchor_importance_dgcnn_vs_pool_anchor_normal.csv, type_importance_dgcnn_vs_pool_anchor_normal.csv")
print("Ablation outputs: explanations_dgcnn_comparison_ablation.csv, anchor_importance_summary_ablation.csv,")
print("                  type_importance_summary_ablation.csv, type_importance_top5_ablation.csv,")
print("                  anchor_importance_dgcnn_vs_pool_anchor_ablation.csv, type_importance_dgcnn_vs_pool_anchor_ablation.csv")

## v2 addition: feature-level explanation

Everything above (mirroring `08g`) explains importance at the **node** level ("was this
`building` node important"). This section adds a finer granularity:
**which NAMED FEATURE COMPONENT within that node** actually drove the
prediction -- e.g. was it a building's footprint `area`, its `height`,
or its `type_embed` (categorical building-type embedding) that mattered?

Mechanism (`explain.run_gnnexplainer_features`): the same GNNExplainer
optimization as above, but the learned mask has one value per **named
feature component per node** (see `explain.get_feature_components`)
instead of one value per node -- hooking each encoder's
`_assemble_raw_features` (the raw, pre-`input_proj` concatenated
feature vector) rather than `assemble_inputs`. Continuous fields
(position, area, height, ...) each get their own mask; a categorical
field's embedding block (`class_embed`, `type_embed`, `highway_embed`)
gets ONE shared mask value, since masking individual embedding
dimensions isn't interpretable -- only "was this categorical field
used at all" is.

**Cost note**: this runs a SECOND, independent GNNExplainer
optimization per point (feature-level masks are a different
parameterization from the node-level masks above, not derivable from
them) -- roughly doubles this notebook's total GNNExplainer time. It
reuses the exact same sampled points (same `EXPLAIN_SEED`,
`N_PER_CATEGORY`) as the node/edge-level pass above, so per-point
results are directly comparable across granularities.

**Edge features are NOT included here** -- true per-edge-attribute-
dimension masking (e.g. distinguishing `on_segment`'s distance
component from its highway-type component) would need an analogous
raw/projected split inside `assemble_edge_attrs`, scoped out for now;
edge importance stays at the existing per-edge granularity from the
node/edge-level section above.

### Run feature-level explanation -- normal and ablation, SEPARATELY (reuses the same sampled points)

In [ ]:
def run_feature_explanation_pass(scenarios, use_ablation):
    records = []
    label = "ablation" if use_ablation else "normal"
    for scenario in scenarios:
        print(f"\n=== [features/{label}] Scenario {scenario} ===")
        model, stats, best_repeat, tag = load_scenario_model(scenario, use_ablation=use_ablation)
        points = pick_points_to_explain(tag, best_repeat, n_per_category=N_PER_CATEGORY)
        print(f"  explaining {len(points)} points (feature-level)")

        for p in points:
            pid, cat = p["point_id"], p["category"]
            svg_raw = torch.load(SVG_DIR / f"{pid}.pt", weights_only=False)
            tvg_raw = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
            svg_norm, tvg_norm = ds.apply_normalization(copy.deepcopy(svg_raw), copy.deepcopy(tvg_raw), stats)

            try:
                recs = explain.explain_scenario_point_features(
                    model, scenario, svg_norm, tvg_norm, point_id=pid, category=cat,
                    gnnexplainer_epochs=GNNEXPLAINER_EPOCHS, gnnexplainer_lr=GNNEXPLAINER_LR)
                records.extend(recs)
            except Exception as e:
                print(f"    !! failed to explain (features) {pid} ({cat}): {type(e).__name__}: {e}")
    print(f"\n[features/{label}] total explained (point, branch) records: {len(records)}")
    return records


normal_feature_records = run_feature_explanation_pass(NORMAL_SCENARIOS, use_ablation=False)
ablation_feature_records = run_feature_explanation_pass(ABLATION_SCENARIOS, use_ablation=True)

### Aggregate into two tidy CSVs

In [ ]:
feature_df_normal = explain.aggregate_feature_explanations(normal_feature_records)
feature_df_ablation = explain.aggregate_feature_explanations(ablation_feature_records)

feature_df_normal.to_csv(EXPLAIN_DIR / "feature_explanations_dgcnn_comparison_normal.csv", index=False)
feature_df_ablation.to_csv(EXPLAIN_DIR / "feature_explanations_dgcnn_comparison_ablation.csv", index=False)

print(f"Saved {len(feature_df_normal)} rows to {EXPLAIN_DIR / 'feature_explanations_dgcnn_comparison_normal.csv'}")
print(f"Saved {len(feature_df_ablation)} rows to {EXPLAIN_DIR / 'feature_explanations_dgcnn_comparison_ablation.csv'}")
display(feature_df_normal.head(10))
display(feature_df_ablation.head(10))

### Per-scheme feature-component importance summary

In [ ]:
feature_pivot_normal_df, feature_topn_normal_df = explain.build_feature_importance_report(feature_df_normal, top_n=5)
feature_pivot_ablation_df, feature_topn_ablation_df = explain.build_feature_importance_report(feature_df_ablation, top_n=5)

feature_pivot_normal_df.to_csv(EXPLAIN_DIR / "feature_importance_summary_normal.csv", index=False)
feature_topn_normal_df.to_csv(EXPLAIN_DIR / "feature_importance_top5_normal.csv", index=False)
feature_pivot_ablation_df.to_csv(EXPLAIN_DIR / "feature_importance_summary_ablation.csv", index=False)
feature_topn_ablation_df.to_csv(EXPLAIN_DIR / "feature_importance_top5_ablation.csv", index=False)

print(f"[normal]   saved {len(feature_pivot_normal_df)} summary rows, {len(feature_topn_normal_df)} top-5 rows")
print(f"[ablation] saved {len(feature_pivot_ablation_df)} summary rows, {len(feature_topn_ablation_df)} top-5 rows")
display(feature_topn_normal_df)
display(feature_topn_ablation_df)

## Figure: top-5 feature-component importance per scenario, seaborn, L-shaped axes

Same convention as the node/edge-level figures above -- `sns.despine()`
(L-shaped, only left+bottom spines), muted palette. Each bar is one
`node_type.feature` pair (e.g. `building.height`, `signage.class_embed`),
not a whole node type.

In [ ]:
def plot_top_feature_importance_grid(topn_df, title, filename):
    scenarios = sorted(topn_df["scenario"].unique())
    n = len(scenarios)
    if n == 0:
        print(f"No rows in {title} -- skipping figure.")
        return
    ncols = 3
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), squeeze=False)

    for i, scen in enumerate(scenarios):
        ax = axes[i // ncols][i % ncols]
        scen_df = topn_df[topn_df["scenario"] == scen].sort_values("mean_importance")
        labels = [f"{r.node_type}.{r.feature}" for r in scen_df.itertuples()]
        ax.barh(labels, scen_df["mean_importance"], color=PALETTE[1])
        ax.set_title(scen, fontsize=11, fontweight="bold")
        ax.set_xlim(0, 1)
        ax.set_xlabel("mean gnnexplainer importance (feature-level)")
        ax.tick_params(axis="y", labelsize=8)
        sns.despine(ax=ax)

    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches="tight")
    print(f"Saved figure to {FIG_DIR / filename}")
    plt.show()


plot_top_feature_importance_grid(feature_topn_normal_df, "Top-5 feature-component importance per scenario (normal)",
                                  "fig_top_feature_importance_normal.png")
plot_top_feature_importance_grid(feature_topn_ablation_df, "Top-5 feature-component importance per scenario (ablation)",
                                  "fig_top_feature_importance_ablation.png")

In [ ]:
print("08i_v2 feature-level explainability run complete.")
print(f"[normal]   {feature_df_normal['point_id'].nunique()} unique points, "
      f"{feature_df_normal['scenario'].nunique()} scenario/branch tags, "
      f"{feature_df_normal['feature'].nunique()} distinct feature components explained.")
print(f"[ablation] {feature_df_ablation['point_id'].nunique()} unique points, "
      f"{feature_df_ablation['scenario'].nunique()} scenario/branch tags.")
print()
print("New (v2) outputs, normal:   feature_explanations_dgcnn_comparison_normal.csv,")
print("                            feature_importance_summary_normal.csv, feature_importance_top5_normal.csv")
print("New (v2) outputs, ablation: feature_explanations_dgcnn_comparison_ablation.csv,")
print("                            feature_importance_summary_ablation.csv, feature_importance_top5_ablation.csv")
print("(node/edge-level outputs from the section above are unchanged from 08i)")